# Language Modelling Lab (week 3)
This notebook provides the "starter" code in the week 3 lab.  You should refer to the pdf document for what to do in this lab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/anlp_notebooks/lab3

/content/drive/MyDrive/anlp_notebooks/lab3


## 1 Getting Started
We need to get the names of files in the training directory and split them into training and testing 50:50.

In [ ]:
import os,random,math
TRAINING_DIR="sentence-completion/Holmes_Training_Data"  #this needs to be the parent directory for the training corpus

def get_training_testing(training_dir=TRAINING_DIR, split=0.5):

    filenames=os.listdir(training_dir)
    n=len(filenames)
    print("There are {} files in the training directory: {}".format(n,training_dir))
    random.seed(53)  #if you want the same random split every time
    random.shuffle(filenames)
    index=int(n*split)
    return(filenames[:index],filenames[index:])

trainingfiles,heldoutfiles=get_training_testing()


There are 523 files in the training directory: sentence-completion/Holmes_Training_Data


In [ ]:
len(trainingfiles)

261

## 2  Building a unigram model

THe code below implements a simple unigram model.  The class stores the unigram probability distribution and provides methods for training and lookup.

In [ ]:
from nltk import word_tokenize as tokenize
import operator

class language_model():

    def __init__(self,trainingdir=TRAINING_DIR,files=[]):
        #store the names of the files containing training data and run the training method
        self.training_dir=trainingdir
        self.files=files
        self.train()

    def train(self):
        #initialise an empty dictionary which will be the unigram model {w:P(w)} when training is complete
        self.unigram={}
        #process all of the training data, accumulating counts of events
        self._processfiles()
        #convert the accumulated counts to probabilities
        print("Finalising probability distribution")
        self._convert_to_probs()

    def _processline(self,line):
        #process each line of a file
        #each line is tokenized and has a special start and end token added
        #counts of tokens are added to the self.unigram count model
        tokens=["__START"]+tokenize(line)+["__END"]
        for token in tokens:
            self.unigram[token]=self.unigram.get(token,0)+1


    def _processfiles(self):
        #process each file in turn
        for afile in self.files:
            print("Processing {}".format(afile))
            with open(os.path.join(self.training_dir,afile),errors='ignore') as instream:
                    for line in instream:
                        line=line.rstrip()
                        if len(line)>0:
                            self._processline(line)


    def _convert_to_probs(self):
        #self.unigram initially counts counts for each token {token:freq(token)}
        #sum all of the frequencies and divide each frequency by that sum to get probabilities

        self.unigram={k:v/sum(self.unigram.values()) for (k,v) in self.unigram.items()}

    def get_prob(self,token,method="unigram"):
        #simple look up method
        if method=="unigram":
            return self.unigram.get(token,0)
        else:
            print("Not implemented: {}".format(method))
            return 0






In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
MAX_FILES=5
mylm=language_model(files=trainingfiles[:MAX_FILES])

Processing DYNMT10.TXT
Processing 04TOM10.TXT
Processing PROTT10.TXT
Processing NWIND10.TXT
Processing BCPTV10.TXT
Finalising probability distribution


Make sure you look up some probabilities of words in your model.  Pick some words which you would expect to have high probabilities and some words which you would expect to have low probabilities.

As an extension, see how these change if you use a bigger portion of the training data to train your model.


In [ ]:
mylm.get_prob("man")

0.0016016977996676477

In [ ]:
mylm.get_prob("the")

0.03971459747332172

In [ ]:
mylm.get_prob("and")

0.02225358880413238

In [ ]:
mylm.get_prob("amazing")

5.005305623961399e-06

## 2.2 Generation
Add some functionality to your class so that you can generate a string of highly probably words.

Refer to the pdf document for tips on how to do this.

**How to sample a random number from a probability distribution in python**



In [ ]:
import random

words = list(dist.keys())
probs = list(dist.values())

sample = random.choices(words, weights=probs, k=1)[0]
print(sample)

# Numpy version
sample = np.random.choice(words, p=probs)
print(sample)

1) “Top-k then pick randomly among them”

Step: sort words by probability, take the top k most probable words.

Then: choose uniformly from those k words (or sometimes weighted inside top-k, but your text says “randomly choosing 1 of the top k” → usually uniform).

Effect: you almost always generate high-probability/common words. Rare words never appear (unless they’re in top-k).

So it’s like: “Only allow the model to speak using its favourite k words.”

2) “Assign ranges proportional to probability” (CDF sampling)

This samples from the full distribution.

Every word can be chosen, but high-prob words get picked more often because they have bigger ranges.

Effect: more variety; still respects probabilities; rare words can appear sometimes.

So it’s like: “Spin a weighted wheel where slice sizes = probabilities.”

In [ ]:
def _sample_words(self, word_dist):
  import random
  x = random.random()

  for word, start, end in word_dist:
    if start <= x < end:
      return word

def _cdf_sampling(self):

  word_dist = []
  for word, prob in self.unigram.items():

    if not word_dist:
      word_dist.append((word, 0.000, prob))

    else:
      word_dist.append((word, word_dist[-1][2], word_dist[-1][2] + prob))

  return word_dist

def generate_sentence(self, max_length=20):

    word_dist = self._cdf_sampling()
    sentence = []

    for _ in range(max_length):

        word = self._sample_words(word_dist)

        if word in ["__START"]:
            continue

        if word == "__END":
            break

        sentence.append(word)

    return " ".join(sentence)



language_model._sample_words = _sample_words
language_model._cdf_sampling = _cdf_sampling
language_model.generate_sentence = generate_sentence

In [ ]:
mylm.generate_sentence()

'a . attention it The accidents like carried lady for mingled and nothing glided his at Apples'

**In a unigram model:**

each word is chosen independently from the same overall word distribution.

So the model knows things like:

- “the” is common

- “and” is common

- “it” is common

- “attention” appeared in training

- “Apples” appeared in training

But it does not know:

- what usually comes after “The”

- that “a .” is odd

- that “his at” is ungrammatical

- that “Apples” in the end is random here

So it generates:

- locally random

- globally incoherent

- sometimes with real words, but no syntax

Punctuation appears oddly, Because . is just another token in your unigram distribution.

It has no rule saying:

- punctuation should come after a complete clause

- sentence should stop right after .

- next word after . should start a new sentence

Unless you explicitly code that logic.

We need a logic, because otherwise the model may sample . in the middle and then continue.

In [ ]:
def generate_sentence(self, max_length=20):
    word_dist = self._cdf_sampling()
    sentence = []

    for _ in range(max_length):
        word = self._sample_words(word_dist)

        if word == "__START":
            continue
        if word == "__END":
            break

        sentence.append(word)

        # To stop sentences if one of these occur like in real world
        if word in [".", "!", "?"]:
            break

    return " ".join(sentence)

language_model.generate_sentence = generate_sentence

In [ ]:
mylm.generate_sentence()

'refund the As as .'

## 3 Adding Bigrams

Refer to the pdf document

In [ ]:
sentence= "When did the cat sit on the mat?"

sent_dict ={}

tokens_sent=["__START"]+tokenize(sentence)+["__END"]
previous="__END"

for token in tokens_sent:
  current=sent_dict.get(previous,{})
  current[token]=current.get(token,0)+1
  sent_dict[previous]=current
  previous=token

sent_dict


#sent_dict

{'__END': {'__START': 1},
 '__START': {'When': 1},
 'When': {'did': 1},
 'did': {'the': 1},
 'the': {'cat': 1, 'mat': 1},
 'cat': {'sit': 1},
 'sit': {'on': 1},
 'on': {'the': 1},
 'mat': {'?': 1},
 '?': {'__END': 1}}

In [ ]:
bigram = {}

tokens=["__START"]+tokenize(sentence)+["__END"]

for i in range(len(tokens) - 1):

    prev_word = tokens[i]
    curr_word = tokens[i+1]

    if prev_word not in bigram:
        bigram[prev_word] = {}

    bigram[prev_word][curr_word] = bigram[prev_word].get(curr_word, 0) + 1

bigram

{'__START': {'When': 1},
 'When': {'did': 1},
 'did': {'the': 1},
 'the': {'cat': 1, 'mat': 1},
 'cat': {'sit': 1},
 'sit': {'on': 1},
 'on': {'the': 1},
 'mat': {'?': 1},
 '?': {'__END': 1}}

In [ ]:
from nltk import word_tokenize as tokenize
import operator

class language_model():

    def __init__(self,trainingdir=TRAINING_DIR,files=[]):
        #store the names of the files containing training data and run the training method
        self.training_dir=trainingdir
        self.files=files
        self.train()

    def train(self):
        #initialise an empty dictionary which will be the unigram model {w:P(w)} when training is complete
        self.unigram={}
        self.bigram={}
        #process all of the training data, accumulating counts of events
        self._processfiles()
        #convert the accumulated counts to probabilities
        print("Finalising probability distribution")
        self._convert_to_probs()

    def _processline(self,line):
        #process each line of a file
        #each line is tokenized and has a special start and end token added
        #counts of tokens are added to the self.unigram count model
        tokens=["__START"]+tokenize(line)+["__END"]
        for token in tokens:
            self.unigram[token]=self.unigram.get(token,0)+1

        for i in range(len(tokens) - 1):

          prev_word = tokens[i]
          curr_word = tokens[i+1]

          if prev_word not in self.bigram:
              self.bigram[prev_word] = {}

          self.bigram[prev_word][curr_word] = self.bigram[prev_word].get(curr_word, 0) + 1


    def _processfiles(self):
        #process each file in turn
        for afile in self.files:
            print("Processing {}".format(afile))
            with open(os.path.join(self.training_dir,afile),errors='ignore') as instream:
                    for line in instream:
                        line=line.rstrip()
                        if len(line)>0:
                            self._processline(line)


    def _convert_to_probs(self):
        #self.unigram initially counts counts for each token {token:freq(token)}
        #sum all of the frequencies and divide each frequency by that sum to get probabilities

        self.unigram={k:v/sum(self.unigram.values()) for (k,v) in self.unigram.items()}
        self.bigram={key:{k:v/sum(adict.values()) for (k,v) in adict.items()} for (key,adict) in self.bigram.items()}

    def get_prob(self,token,context=None,method="unigram"):
        #simple look up method
        if method=="unigram":
            return self.unigram.get(token,0)
        elif method == "bigram":
            if not context:
                return 0
            return self.bigram.get(context[-1], {}).get(token, 0)

    def _sample_words(self, word_dist):
      import random
      x = random.random()

      for word, start, end in word_dist:
        if start <= x < end:
          return word

    def _cdf_sampling(self, method="unigram", prev_word=None):

      word_dist = []
      if method == "unigram":
        source_dist = self.unigram

      elif method == "bigram":
        source_dist = self.bigram.get(prev_word, {})


      cumulative = 0.0
      for word, prob in source_dist.items():
          start = cumulative
          cumulative += prob
          end = cumulative
          word_dist.append((word, start, end))


      return word_dist

    def generate_sentence(self, max_length=20, method="unigram"):

        sentence = []

        if method == "unigram":
            word_dist = self._cdf_sampling(method="unigram")

            for _ in range(max_length):
                word = self._sample_words(word_dist)

                if word == "__START":
                    continue
                if word == "__END":
                    break

                sentence.append(word)

                if word in [".", "!", "?"]:
                    break

        elif method == "bigram":
            prev_word = "__START"

            for _ in range(max_length):
                word_dist = self._cdf_sampling(method="bigram", prev_word=prev_word)

                if not word_dist:
                    break

                word = self._sample_words(word_dist)

                if word == "__END":
                    break

                sentence.append(word)
                prev_word = word

                if word in [".", "!", "?"]:
                    break

        return " ".join(sentence)

In [ ]:
MAX_FILES=5
mylm_bi=language_model(files=trainingfiles[:MAX_FILES])

Processing DYNMT10.TXT
Processing 04TOM10.TXT
Processing PROTT10.TXT
Processing NWIND10.TXT
Processing BCPTV10.TXT
Finalising probability distribution


In [ ]:
mylm_bi.bigram

{'__START': {'The': 0.01367372172521638,
  'by': 0.0017092152156520475,
  '#': 0.00014546512473634447,
  'Copyright': 0.00014546512473634447,
  'the': 0.032875118190413846,
  'Please': 0.00040002909302494727,
  'We': 0.0016728489344679613,
  'electronic': 0.00018183140592043057,
  '*': 0.0021456105898610806,
  'Information': 0.00036366281184086114,
  'further': 0.00036366281184086114,
  'September': 3.636628118408612e-05,
  'Corrected': 0.00018183140592043057,
  'VERSIONS': 0.00018183140592043057,
  'of': 0.015782966033893373,
  'midnight': 0.0002545639682886028,
  'Midnight': 0.00018183140592043057,
  'preliminary': 0.00018183140592043057,
  'and': 0.03182049603607535,
  'up': 0.0011637209978907557,
  'in': 0.00814604698523529,
  'a': 0.008582442359444322,
  'look': 0.00040002909302494727,
  'new': 0.0004363953742090334,
  'fifty': 0.0002181976871045167,
  'to': 0.013819186849952723,
  'searched': 0.00018183140592043057,
  'projected': 0.0002545639682886028,
  'per': 0.000218197687104

In [ ]:
mylm_bi.get_prob("man",context="an",method="bigram")

0

In [ ]:
mylm_bi.generate_sentence(method="bigram")

'eyes , tell you keep this dishonourable death were both parties on the bottom of the collar ; `` Wait'

In [ ]:
word_dist = mylm_bi._cdf_sampling(method="bigram", prev_word="__START")
mylm._sample_words(word_dist)

'and'

## 4 Perplexity

Refer to the pdf document

In [ ]:
def compute_prob_line(self,line,method="unigram"):
    #this will add _start to the beginning of a line of text
    #compute the probability of the line according to the desired model
    #and returns log probability together with number of tokens

    tokens=["__START"]+tokenize(line)+["__END"]
    acc=0
    for i,token in enumerate(tokens[1:]):
        acc+=math.log(self.get_prob(token,tokens[:i+1],method))
    return acc,len(tokens[1:])


def compute_probability(self,filenames=[],method="unigram"):
    #computes the log probability (and length) of a corpus contained in filenames
    if filenames==[]:
        filenames=self.files

    total_p=0
    total_N=0
    for i,afile in enumerate(filenames):
        print("Processing file {}:{}".format(i,afile))
        try:
            with open(os.path.join(self.training_dir,afile)) as instream:
                for line in instream:
                    line=line.rstrip()
                    if len(line)>0:
                        p,N=self.compute_prob_line(line,method=method)
                        total_p+=p
                        total_N+=N
        except UnicodeDecodeError:
            print("UnicodeDecodeError processing file {}: ignoring rest of file".format(afile))
    return total_p,total_N

def compute_perplexity(self,filenames=[],method="unigram"):

    #compute the log probability and length of the corpus
    #calculate perplexity
    #lower perplexity means that the model better explains the data

    p,N=self.compute_probability(filenames=filenames,method=method)
    #print(p,N)
    pp=math.exp(-p/N)
    return pp

language_model.compute_prob_line = compute_prob_line
language_model.compute_probability = compute_probability
language_model.compute_perplexity = compute_perplexity


In [ ]:
mylm=language_model(files=trainingfiles[:MAX_FILES])

Processing DYNMT10.TXT
Processing 04TOM10.TXT
Processing PROTT10.TXT
Processing NWIND10.TXT
Processing BCPTV10.TXT
Finalising probability distribution


In [ ]:
mylm.generate_sentence(method='bigram')

"' a bit and when with such announcement ."

In [ ]:
mylm.compute_perplexity()

Processing file 0:DYNMT10.TXT
Processing file 1:04TOM10.TXT
Processing file 2:PROTT10.TXT
Processing file 3:NWIND10.TXT
Processing file 4:BCPTV10.TXT


515.0580261721353

In [ ]:
mylm.compute_perplexity(method='bigram')

Processing file 0:DYNMT10.TXT
Processing file 1:04TOM10.TXT
Processing file 2:PROTT10.TXT
Processing file 3:NWIND10.TXT
Processing file 4:BCPTV10.TXT


49.38983114902966

But there is one major problem

If any token has probability 0, then: $math.log(0)$ will crash or produce an error.

So perplexity code is mathematically correct, but unsmoothed models will fail on unseen words/bigrams.

For real evaluation, you usually need smoothing.

### 5 Dealing with Unknowns
What happens when you compute the perplexity of a corpus not used in training?
Zero probabilities ....


In [ ]:
mylm.compute_perplexity(filenames=heldoutfiles[:MAX_FILES])

Processing file 0:GABRM10.TXT


ValueError: math domain error

In [ ]:
mylm.unigram

{'__START': 0.06881794702384528,
 'The': 0.002665325244759445,
 'Project': 0.00030282099024966463,
 'Gutenberg': 0.00018019100246261037,
 'Etext': 6.256632029951748e-05,
 'of': 0.018572186517708773,
 'Dynamiter': 1.5015916871884198e-05,
 ',': 0.060023625042545097,
 'by': 0.002532684645724468,
 'Stevensons': 5.005305623961399e-06,
 '__END': 0.06881794702384528,
 'Robert': 2.0021222495845597e-05,
 'Louis': 2.2523875307826296e-05,
 'Stevenson': 3.7539792179710495e-05,
 'and': 0.02225358880413238,
 'Fanny': 1.0010611247922799e-05,
 'Van': 1.0010611247922799e-05,
 'De': 1.0010611247922799e-05,
 'Grift': 1.0010611247922799e-05,
 '#': 2.7529180931787693e-05,
 '32': 2.5026528119806996e-06,
 'in': 0.00944751436522714,
 'our': 0.0007307746210983643,
 'series': 4.755040342763329e-05,
 'Copyright': 1.0010611247922799e-05,
 'laws': 3.0031833743768396e-05,
 'are': 0.0018519630808657176,
 'changing': 2.2523875307826296e-05,
 'all': 0.0027454101347428274,
 'over': 0.0007858329829619397,
 'the': 0.0397

In [ ]:
rarewords=[]
for k,v in mylm.unigram.items():
    if v < 0.05:
        rarewords.append(k)

for word in rarewords:
    del mylm.unigram[word]

In [ ]:
mylm.unigram

{'__START': 0.06881794702384528,
 ',': 0.060023625042545097,
 '__END': 0.06881794702384528}

In [ ]:
def make_unknowns(self,known=2):
    unknown=0
    for (k,v) in list(self.unigram.items()):
        if v<known:
            del self.unigram[k]
            self.unigram["__UNK"]=self.unigram.get("__UNK",0)+v
    for (k,adict) in list(self.bigram.items()):
        for (kk,v) in list(adict.items()):
            isknown=self.unigram.get(kk,0)
            if isknown==0:
                adict["__UNK"]=adict.get("__UNK",0)+v
                del adict[kk]
        isknown=self.unigram.get(k,0)
        if isknown==0:
            del self.bigram[k]
            current=self.bigram.get("__UNK",{})
            current.update(adict)
            self.bigram["__UNK"]=current

        else:
            self.bigram[k]=adict

language_model.make_unknowns = make_unknowns

In [ ]:
def train(self):
    self.unigram={}
    self.bigram={}

    self._processfiles()
    print("Removing rare words")
    self.make_unknowns()
    print("Finalising probability distribution")

    self._convert_to_probs()

language_model.train = train

In [ ]:
mylm=language_model(files=trainingfiles[:MAX_FILES])
p=mylm.compute_perplexity()
print("Training data unigram perplexity: {}".format(p))
p=mylm.compute_perplexity(filenames=heldoutfiles[:MAX_FILES])
print("Testing data unigram perplexity: {}".format(p))

Processing DYNMT10.TXT
Processing 04TOM10.TXT
Processing PROTT10.TXT
Processing NWIND10.TXT
Processing BCPTV10.TXT
Removing rare words
Finalising probability distribution
Processing file 0:DYNMT10.TXT


ValueError: math domain error

## 6 Smoothing

Refer to the pdf document

## 7 Extensions